In [1]:
from pydrake.all import (RobotDiagramBuilder,
                         LoadModelDirectives,
                         ProcessModelDirectives,
                         AddDefaultVisualization,
                         ApplyVisualizationConfig,
                         VisualizationConfig,
                         Rgba, 
                         SceneGraphCollisionChecker,
                         StartMeshcat,
                         IrisZo, 
                         IrisZoOptions, 
                         IrisInConfigurationSpace, 
                         IrisOptions,
                         HPolyhedron,
                         Hyperellipsoid)
import time
import numpy as np
import yaml
import os
root = '../'
directives_file = root+'assets/directives/iiwa7_on_table.yaml'
meshcat = StartMeshcat()
builder = RobotDiagramBuilder()
plant = builder.plant()
scene_graph = builder.scene_graph()
parser = builder.parser()

parser.package_map().Add("adaptive_decomp", root+"assets")
parser.package_map().Add("iiwa_description", root+"assets/iiwa")
parser.package_map().Add("wsg_description", root+"assets/wsg_description")
parser.package_map().Add("tri_finray_gripper", root+"assets/tri_finray_gripper")

directives = LoadModelDirectives(directives_file)
models = ProcessModelDirectives(directives, plant, parser)
plant.Finalize()

config = VisualizationConfig()
config.enable_alpha_sliders = False
config.publish_contacts=False
config.publish_inertia = False
config.default_proximity_color = Rgba(0.8,0,0,0.5)
ApplyVisualizationConfig(config, builder.builder(), meshcat=meshcat)
# visualizer = AddDefaultVisualization(builder.builder(), meshcat)
diagram = builder.Build()
diagram_context = diagram.CreateDefaultContext()
plant_context = plant.GetMyContextFromRoot(diagram_context)
diagram.ForcedPublish(diagram_context)
robot_model_instances = [plant.GetModelInstanceByName(m) for m in ['iiwa7', 'wsg', 'finray']]

checker = SceneGraphCollisionChecker(model = diagram, 
                                     robot_model_instances = robot_model_instances,
                                     edge_step_size = 0.05)
meshcat.SetProperty('/drake/proximity', "visible", True)

INFO:drake:Meshcat listening for connections at http://localhost:7000
INFO:drake:Allocating contexts to support implicit context parallelism 20


In [2]:
from cspace_utils.plotting import plot_triad
from pydrake.all import ModelInstanceIndex
X = plant.EvalBodyPoseInWorld(plant_context, plant.GetBodyByName('camera_box_link', ModelInstanceIndex(6)))
plot_triad(X,meshcat)

In [3]:
import sys
PYCUCI_ROOT = "/home/peter/gitcspace/cuciv0"
sys.path.append(PYCUCI_ROOT+'/bazel-bin/cuci/src/pybind/pycuci')
import pycuci_bindings as cci

parser.package_map().Add("adaptive_decomp", root+"assets")
parser.package_map().Add("iiwa_description", root+"assets/iiwa")
parser.package_map().Add("wsg_description", root+"assets/wsg_description")
parser.package_map().Add("tri_finray_gripper", root+"assets/tri_finray_gripper")

cci_parser = cci.URDFParser()
cci_parser.register_package("adaptive_decomp", root+"assets")
cci_parser.register_package("iiwa_description", root+"assets/iiwa")
cci_parser.register_package("wsg_description", root+"assets/wsg_description")
cci_parser.register_package("tri_finray_gripper", root+"assets/tri_finray_gripper")
cci_parser.parse_directives("../assets/directives/iiwa7_on_table.yaml")
cci_plant = cci_parser.build_plant()
cci_mplant = cci_plant.getMinimalPlant()
cci_domain = cci.HPolyhedron()
cci_domain.MakeBox(cci_plant.getPositionLowerLimits(), cci_plant.getPositionUpperLimits())
cci_fei_opts = cci.FastEdgeInflationOptions()
cci_fei_opts.num_particles = 10000
cci_fei_opts.max_hyperplanes_per_iteration = 30
edge_inflator = cci.CudaEdgeInflator(cci_plant.getMinimalPlant(), 
                                     cci_plant.getRobotGeometryIds(), cci_fei_opts, cci_domain)

Successfully registered package 'adaptive_decomp' with path: ../assets
Successfully registered package 'iiwa_description' with path: ../assets/iiwa
Successfully registered package 'wsg_description' with path: ../assets/wsg_description
Successfully registered package 'tri_finray_gripper' with path: ../assets/tri_finray_gripper


In [4]:
samps = cci.UniformSampleInHPolyhedraCuda([cci_domain], 
                                          cci_domain.ChebyshevCenter(), 
                                          4000, 
                                          50)[0]
col_free_cci = cci.CheckCollisionFreeCuda(samps, cci_mplant)
col_free_drake = checker.CheckConfigsCollisionFree(samps.T, parallelize=True)

diff = 1.*np.array(col_free_drake)- 1.*np.array(col_free_cci)
print(np.any(diff))
print(np.where(diff))

execution time cuda no copy: 7 ms
False
(array([], dtype=int64),)


In [5]:
# mismatch = samps[:, np.where(diff)[0]]

# for c in [(mismatch.T)[0]]:
#     plant.SetPositions(plant_context, c)
#     diagram.ForcedPublish(diagram_context)
#     print(f"drake { checker.CheckConfigCollisionFree(c)}")
#     print(f"cci { cci.CheckCollisionFree(c, cci_mplant)}")
#     # time.sleep(3)

In [6]:
from cspace_utils.plotting import plot_points
min_corner = np.array([-0.4, -1.25, 0.])
max_corner = np.array([1.1, 1.25, 1.])

ws_corners_online_voxels = np.array([
    [min_corner[0], min_corner[1], min_corner[2]],
    [min_corner[0], min_corner[1], max_corner[2]],
    [min_corner[0], max_corner[1], min_corner[2]],
    [min_corner[0], max_corner[1], max_corner[2]],
    [max_corner[0], min_corner[1], min_corner[2]],
    [max_corner[0], min_corner[1], max_corner[2]],
    [max_corner[0], max_corner[1], min_corner[2]],
    [max_corner[0], max_corner[1], max_corner[2]]
])

plot_points(meshcat, 
            ws_corners_online_voxels, 
            'drm/online_voxel_range_markers', 
            0.03,
            Rgba(1,0,1,0.9))

In [7]:
import ipywidgets as widgets
from functools import partial

q = np.zeros(plant.num_positions()) 
plant.SetPositions(plant_context, np.ones(7))
diagram.ForcedPublish(diagram_context)

sliders = []
for i in range(plant.num_positions()):
    q_low = plant.GetPositionLowerLimits()[i]*0.99
    q_high = plant.GetPositionUpperLimits()[i]*0.99
    sliders.append(widgets.FloatSlider(min=q_low, max=q_high, value=(q_high+q_low)/2, step=0.001, description=f"q{i}"))

def handle_slider_change(change, idx):
    q[idx] = change['new']
    plant.SetPositions(plant_context, q)
    if not checker.CheckConfigCollisionFree(q):
        print('collision')
    diagram.ForcedPublish(diagram_context)
    
idx = 0
for slider in sliders:
    slider.observe(partial(handle_slider_change, idx = idx), names='value')
    idx+=1

for slider in sliders:
    display(slider)

FloatSlider(value=0.0, description='q0', max=2.9373894, min=-2.9373894, step=0.001)

FloatSlider(value=0.0, description='q1', max=2.07345105, min=-2.07345105, step=0.001)

FloatSlider(value=0.0, description='q2', max=2.9373894, min=-2.9373894, step=0.001)

FloatSlider(value=0.0, description='q3', max=2.07345105, min=-2.07345105, step=0.001)

FloatSlider(value=0.0, description='q4', max=2.9373894, min=-2.9373894, step=0.001)

FloatSlider(value=0.0, description='q5', max=2.07345105, min=-2.07345105, step=0.001)

FloatSlider(value=0.0, description='q6', max=3.02378274, min=-3.02378274, step=0.001)

In [8]:
scene_graph_context = scene_graph.GetMyContextFromRoot(diagram_context)
query_object = scene_graph.get_query_output_port().Eval(scene_graph_context)
inspector = query_object.inspector()

In [10]:
plant.SetPositions(plant_context, np.zeros(7))
diagram.ForcedPublish(diagram_context)

In [13]:
from cspace_utils.plotting import plot_triad
from pydrake.all import RigidTransform
X = plant.EvalBodyPoseInWorld(plant_context, plant.GetBodyByName('iiwa_link_ee') )
plot_triad(X,meshcat, size = 0.1)
# X2 = RigidTransform(X.rotation(),  X.translation()+ np.array([0, 0, 0.28]))
# plot_triad(X2,meshcat, size = 0.08, name='tf2')

In [ ]:
frame_ids = inspector.GetAllFrameIds()
for id in frame_ids:
    print(inspector.GetName(id))


In [14]:
# frame_ids = inspector.GetAllFrameIds()
# for id in frame_ids:
#     print(inspector.GetName(id))

link_names = [
"iiwa7::iiwa_link_0",
"iiwa7::iiwa_link_1",
"iiwa7::iiwa_link_2",
"iiwa7::iiwa_link_3",
"iiwa7::iiwa_link_4",
"iiwa7::iiwa_link_5",
"iiwa7::iiwa_link_6",
"iiwa7::iiwa_link_7",
"iiwa7::iiwa_link_ee",
]

In [16]:
from pydrake.all import Rgba, Role, RigidTransform, Sphere
meshcat_prefix = 'test/'
meshcat.Delete(f'{meshcat_prefix}')
meshcat.SetProperty('/drake/illustration', "visible", True)
meshcat.SetProperty('/drake/proximity', "visible", False)
link_name = link_names[0]
draw_world = False
q = np.zeros(plant.num_positions()) 
q = np.zeros(7)
q[-1] = 0.7
plant.SetPositions(plant_context, q)
diagram.ForcedPublish(diagram_context)

rgba = Rgba(0.7, 0.0, 0.7, 0.5)
role = Role.kIllustration
# This is a minimal replication of the work done in MeshcatVisualizer.

for frame_id in inspector.GetAllFrameIds():
    if frame_id == inspector.world_frame_id():
        if not draw_world:
            continue
        frame_path = meshcat_prefix
    else:
        frame_path = f"{meshcat_prefix}/{inspector.GetName(frame_id)}"
    frame_path.replace("::", "/")
    frame_has_any_geometry = False
    # print(inspector.GetName(frame_id) )
    if inspector.GetName(frame_id) != link_name:
        continue
    for geom_id in inspector.GetGeometries(frame_id, role):
        path = f"{frame_path}/{geom_id.get_value()}"
        path.replace("::", "/")
        meshcat.SetObject(path, inspector.GetShape(geom_id), rgba)
        meshcat.SetTransform(path, inspector.GetPoseInFrame(geom_id))
        frame_has_any_geometry = True

    if frame_has_any_geometry:
        X_WF = query_object.GetPoseInWorld(frame_id)
        meshcat.SetTransform(frame_path, X_WF)

In [ ]:
sliders = []
names = ['x', 'y', 'z', 'r']
low = [-0.2, -0.2, 0, 0.01]
high = [0.2, 0.2, 1.2, 0.2]
sph = (np.array(low) + np.array(high))/2
spheres = []
prefix = 'spheres'
color = Rgba(0.2,1,0.2, 0.7)
color2 = Rgba(1,0.2,0.2, 0.7)
mod_link_name = link_name.split('::')[-1]
meshcat.Delete(prefix)


def draw_current_spheres():
    X_WL = plant.EvalBodyPoseInWorld(plant_context, plant.GetBodyByName(mod_link_name))
    id = 0
    for (t_LS, r) in spheres:
        X_WS = X_WL@RigidTransform(t_LS)
        name = prefix + f'/sph_{id}'
        meshcat.SetObject(name, Sphere(r), color)
        meshcat.SetTransform(name, X_WS)
        id = id+1

def print_results(button):
    print(link_name)

    for id, (t_LS, r) in enumerate(spheres):
        string = f'''  <collision name="{mod_link_name}_sph_{id}">
    <geometry>
    <sphere radius="{r:.3f}"/>
    </geometry>
    <origin rpy="0 0 0" xyz="{t_LS[0]:.3f} {t_LS[1]:.3f} {t_LS[2]:.3f}"/>
  </collision>'''
        print(string)

def add_sphere(button):
    X_WS = RigidTransform(np.array(sph[0:3]))
    X_WL = plant.EvalBodyPoseInWorld(plant_context, plant.GetBodyByName(mod_link_name))
    X_LS = X_WL.inverse() @ X_WS
    translation = X_LS.translation()
    spheres.append((translation, sph[-1]))
    draw_current_spheres()

def rem_sphere(button):
    meshcat.Delete(prefix + f'/sph_{len(spheres)-1}')
    spheres.pop(-1)
    draw_current_spheres()

for n, min, max in zip(names, low, high):
    sliders.append(widgets.FloatSlider(min=min, max=max, value=0, step=0.001, description=n))

button = widgets.Button(description="add sphere")

# Attach the function to the button's click event
button.on_click(add_sphere)
display(button)

button2 = widgets.Button(description="remove sphere")

# Attach the function to the button's click event
button2.on_click(rem_sphere)
display(button2)

button3 = widgets.Button(description="print results")

# Attach the function to the button's click event
button3.on_click(print_results)
display(button3)

def handle_slider_change2(change, idx):
    sph[idx] = change['new']
    plant.SetPositions(plant_context, q)
    # checker.CheckConfigCollisionFree(0*np.ones(7))
    diagram.ForcedPublish(diagram_context)
    name2 = prefix + f'/sph_new'
    meshcat.Delete(name2)
    meshcat.SetObject(name2, Sphere(sph[-1]), color2)
    meshcat.SetTransform(name2, RigidTransform(sph[:3]))

idx = 0
for slider in sliders:
    slider.observe(partial(handle_slider_change2, idx = idx), names='value')
    idx+=1

for slider in sliders:
    display(slider)

In [ ]:
len(spheres)

In [18]:
inspector = query_object.inspector()
frame_ids = inspector.GetAllFrameIds()
b = frame_ids[0]

In [48]:
frame_ids = inspector.GetAllFrameIds()
b = frame_ids[0]

In [ ]:
from pydrake.all import Rgba, Role
role = Role.kIllustration
inspector.GetGeometryIdByName(frame_ids[2], role, '' )


In [ ]:
inspector.GetName(frame_ids[0])
gids = inspector.GetGeometries(frame_ids[2])
inspector.GetName(gids[0])

In [ ]:
links = [base, 
         fr3_hand, 
         fr3_hand_tcp, 
         fr3_leftfinger, 
         fr3_link0, 
         fr3_link1, 
         fr3_link2, 
         fr3_link3, 
         fr3_link4, 
         fr3_link5, 
         fr3_link6, 
         fr3_link7, 
         fr3_link8, 
         fr3_rightfinger]



sliders = []


for i in range(plant.num_positions()):
    q_low = plant.GetPositionLowerLimits()[i]*0.99
    q_high = plant.GetPositionUpperLimits()[i]*0.99
    sliders.append(widgets.FloatSlider(min=q_low, max=q_high, value=0, step=0.001, description=f"q{i}"))


In [ ]:
plant.EvalBodyPoseInWorld(plant_context,  
                          plant.GetBodyByName('fr3_linkads1'))

In [39]:
import sys
PYCUCI_ROOT = "/home/peter/gitcspace/cuciv0"
sys.path.append(PYCUCI_ROOT+'/bazel-bin/cuci/src/pybind/pycuci')
import pycuci_bindings as cci



In [ ]:
cci_parser = cci.URDFParser()
cci_parser.register_package("adaptive_decomp", "../../assets/")
cci_parser.register_package("franka_description", "../../assets/franka")
cci_parser.parse_directives("../../assets/directives/fr3_on_table.yaml")
cci_plant = cci_parser.build_plant()
cci_mplant = cci_plant.getMinimalPlant()
cci_domain = cci.HPolyhedron()
cci_domain.MakeBox(cci_plant.getPositionLowerLimits(), cci_plant.getPositionUpperLimits())
cci_fei_opts = cci.FastEdgeInflationOptions()
cci_fei_opts.num_particles = 10000
cci_fei_opts.max_hyperplanes_per_iteration = 30
edge_inflator = cci.CudaEdgeInflator(cci_plant.getMinimalPlant(), 
                                     cci_plant.getRobotGeometryIds(), cci_fei_opts, cci_domain)

In [ ]:
center = cci_domain.ChebyshevCenter()
samps = cci.UniformSampleInHPolyhedraCuda([cci_domain], 
                                          center.reshape(-1,1), 
                                          5000000, 
                                          50)[0]


is_collision_free = cci.CheckCollisionFreeCuda(samps, cci_mplant)

is_collision_free_drake = checker.CheckConfigsCollisionFree(samps.T, 
                                                            parallelize=True)

for i, (a,b) in enumerate(zip(is_collision_free, is_collision_free_drake)):
    assert a==b, f'i = {i}'

In [ ]:
samps.shape